In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = '/content/drive/MyDrive/MIMIC_ICU/'

In [3]:
import pandas as pd

# Path to your dedicated ICU folder
path = '/content/drive/MyDrive/MIMIC_ICU/'

# 1. Load all files from your screenshots
# Note: compression='infer' handles .csv.gz automatically
icu_stays = pd.read_csv(path + 'icustays.csv.gz', compression='infer')
chartevents = pd.read_csv(path + 'chartevents.csv.gz', compression='infer')
inputevents = pd.read_csv(path + 'inputevents.csv.gz', compression='infer')
outputevents = pd.read_csv(path + 'outputevents.csv.gz', compression='infer')
procedureevents = pd.read_csv(path + 'procedureevents.csv.gz', compression='infer')
datetimeevents = pd.read_csv(path + 'datetimeevents.csv.gz', compression='infer')
ingredientevents = pd.read_csv(path + 'ingredientevents.csv.gz', compression='infer')

# 2. Summarize the event tables so we don't create billions of rows
# We use 'stay_id' as the primary key
vitals_sum = chartevents.groupby('stay_id')['valuenum'].agg(['mean', 'max']).reset_index()
vitals_sum.columns = ['stay_id', 'icu_vitals_mean', 'icu_vitals_max']

in_sum = inputevents.groupby('stay_id')['amount'].sum().reset_index().rename(columns={'amount': 'total_input_ml'})
out_sum = outputevents.groupby('stay_id')['value'].sum().reset_index().rename(columns={'value': 'total_urine_out_ml'})
proc_sum = procedureevents.groupby('stay_id')['itemid'].count().reset_index().rename(columns={'itemid': 'proc_count'})
date_sum = datetimeevents.groupby('stay_id')['itemid'].count().reset_index().rename(columns={'itemid': 'date_event_count'})
ingred_sum = ingredientevents.groupby('stay_id')['amount'].sum().reset_index().rename(columns={'amount': 'total_ingred_ml'})

# 3. THE "MAXIMUM EXTRACTION" SEQUENTIAL JOIN
# We use how='outer' to ensure NO ROW is left behind
# We start with icu_stays but keep everything from subsequent tables
master_icu = pd.merge(icu_stays, vitals_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, in_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, out_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, proc_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, date_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, ingred_sum, on='stay_id', how='outer')

# 4. Final Output
print(f"Maximum Extraction Complete.")
print(f"Total rows preserved: {len(master_icu)}")
print(f"Total features collected: {master_icu.shape[1]}")

# View the master datasheet
pd.set_option('display.max_columns', None)
master_icu.head()

Maximum Extraction Complete.
Total rows preserved: 140
Total features collected: 15


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,icu_vitals_mean,icu_vitals_max,total_input_ml,total_urine_out_ml,proc_count,date_event_count,total_ingred_ml
0,10023117,28872262,30057454,Cardiac Vascular Intensive Care Unit (CVICU),Coronary Care Unit (CCU),2171-11-14 10:06:41,2171-11-18 20:49:43,4.446551,44.416088,251.0,10217.479889,11802.0,10.0,299.0,13830.021599
1,10032725,20611640,30101877,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2143-03-22 06:42:00,2143-03-25 15:05:33,3.349687,54.395508,2000.0,3316.333314,5069.0,7.0,55.0,2797.666619
2,10016742,27568122,30425410,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2178-07-22 08:19:00,2178-07-25 16:42:43,3.349803,69.267471,2700.0,7894.762545,3029.0,4.0,43.0,35818.392327
3,10031757,28477280,30458995,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2137-10-12 22:44:57,2137-10-14 17:08:34,1.766400,44.750359,860.0,9835.922232,4215.0,4.0,31.0,16270.844465
4,10022281,29642388,30585761,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2125-06-17 04:12:54,2125-06-18 14:55:55,1.446539,44.861905,239.0,2119.765445,2332.0,3.0,19.0,3849.611259


In [5]:
# Install the excel writer library if not already there
!pip install openpyxl

# Save as Excel
excel_path = '/content/drive/MyDrive/MIMIC_ICU/ICU_Master_Data.xlsx'
master_icu.to_excel(excel_path, index=False)

print(f"Excel file created at: {excel_path}")

Excel file created at: /content/drive/MyDrive/MIMIC_ICU/ICU_Master_Data.xlsx
